# EdgeCrafter — independent CUDA/AMP validation

**Before running:** Colab menu → **Runtime → Change runtime type → GPU**. This notebook uses *only synthetic data* and your public fork. It neither uploads results nor uses corporate resources. A single GPU tests CUDA forward/backward/optimizer in FP32 and AMP; multi-GPU NCCL needs a separate 2-GPU machine.

In [ ]:
import torch, torchvision
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Enable a GPU in Runtime → Change runtime type'
!nvidia-smi

## Install the import/test dependencies and check out the validated branch

Use Colab's existing CUDA-enabled PyTorch; do **not** replace it with CPU-only PyTorch.

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'PyYAML', 'tensorboard', 'pycocotools', 'calflops',
    'transformers==4.46.3', 'scipy==1.16.0', 'opencv-python-headless', 'pytest'], check=True)
repo = '/content/EdgeCrafter'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', '--branch', 'fix/empty-denoising-edge-cases',
                    '--single-branch', 'https://github.com/tvtt2k/EdgeCrafter.git', repo], check=True)
else:
    subprocess.run(['git', '-C', repo, 'checkout', 'fix/empty-denoising-edge-cases'], check=True)
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
print('Commit:')
subprocess.run(['git', '-C', repo, 'rev-parse', '--short', 'HEAD'], check=True)

## Run single-GPU FP32 + AMP validation

Runs the real model/criterion/backward/optimizer on synthetic data, including **normal**, **denoising disabled**, and **zero annotations**. **PASS** means the scenarios work on this GPU; a failure prints a Python traceback.

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, 'tools/validation/gpu_train_smoke.py',
                '--mode', 'single', '--precision', 'both'],
               cwd='/content/EdgeCrafter/ecdetseg', check=True)

## Optional: production-sized 640×640 test

This runs at the default model resolution, query count and decoder depth; it may exceed the memory of smaller free GPUs. Run only if sufficient VRAM is available. **A Colab runtime usually has just one GPU; this does not validate NCCL/DDP.**

In [ ]:
# Uncomment the following lines to attempt a full-configuration pass:
# subprocess.run([sys.executable, 'tools/validation/gpu_train_smoke.py',
#                 '--mode', 'single', '--precision', 'both', '--full-config'],
#                cwd='/content/EdgeCrafter/ecdetseg', check=True)

## Multi-GPU NCCL validation (run separately on a 2-GPU Linux machine)

Copy the commands below into a terminal on a machine with **two visible NVIDIA GPUs**. Colab's standard one-GPU sessions cannot run this. The script checks six iterations with different per-rank numbers of ground-truth objects, compares loss keys, checks gradients, and uses the NCCL backend with a 150-second collective timeout.

```bash
git clone -b fix/empty-denoising-edge-cases https://github.com/tvtt2k/EdgeCrafter.git
cd EdgeCrafter/ecdetseg
# Install this repository's dependencies (use an environment with CUDA-enabled PyTorch).
export CUDA_VISIBLE_DEVICES=0,1
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export NCCL_DEBUG=WARN
PYTHONUNBUFFERED=1 torchrun --standalone --nproc_per_node=2 \
  tools/validation/gpu_train_smoke.py --mode ddp --precision both
```

After running, share the traceback or the `ALL GPU VALIDATION SCENARIOS PASSED` output. Do not publish confidential machine paths or data.